# Suite SVC — The SystemVerilog Constraints dialect

The constraints and distributions of constraint blocks (IEEE 1800 18.5), a dialect of SystemVerilog's family,
`Expressions.SystemVerilog.Constraints`, as mbse-programs has them: its union of its kinds, listing every branch of
SystemVerilog's `Expressions`, and its holder; each kind's shape, its checks and what it binds (a `foreach` its loop
variables, refining `Let`); trees that mix its kinds with expressions, held by `randomize() with` and by an assertion's
operand; invalid syntax; the editions of IEEE 1800 it covers; printing, as mbse-programs' printer writes its kinds; and
evaluation, which refuses each kind by name. The family's `Expressions` and `Types` are suite DIA's.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns"; // its family's new dialects
import * as PythonTypes from "@mbse/expressions/Python/Types";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { NotImplementedError, ValueError } = Errors;
const P = Python3.Expressions;
const [SV, SVV, SVText] = [SystemVerilog.Expressions, SystemVerilog.Simplifiers, SystemVerilog.Text];
const [SVT, SVP, SVA, SVPP] = [SVTypes.Expressions, SVPatterns.Expressions, SVAssertions.Expressions, SVPreprocessor.Expressions];
const [SVC, SVCText] = [SVConstraints.Expressions, SVConstraints.Text];

const store: any = new Builders.Store(); // SystemVerilog's family, after the framework and the standard dialects; Python3 for a foreign term
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, Patterns.install,
  Python3.install, SystemVerilog.install, SVTypes.install, SVPatterns.install, SVAssertions.install,
  SVConstraints.install, SVPreprocessor.install, PythonTypes.install, PythonPatterns.install]) { // Python's new dialects
  store.transaction(install); // after its family's dialect
}
for (const define of [SystemVerilog.define, SVConstraints.define]) { // SystemVerilog's types, which its validation of a cast reads,
  store.transaction(define); // and the editions the Constraints dialect covers
}
const FAMILY: F.Dialect[] = [SV.DIALECT, SVT.DIALECT, SVP.DIALECT, SVA.DIALECT, SVC.DIALECT, SVPP.DIALECT];

/** The holder of the term `spec` writes: the dialect's own `Expression`, written in a transaction. */
function held(spec: unknown, dialect: F.Dialect = SVC.DIALECT): any {
  const holder = dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions).Expression;
  return store.transaction(() => holder((e: any) => e.root(dialect.spec(spec))));
}

/** The term `spec` writes, the root of its dialect's holder. */
function term(spec: unknown, dialect: F.Dialect = SVC.DIALECT): any {
  return F.term(held(spec, dialect));
}

function validate(spec: unknown, bound: string[] = [], dialect: F.Dialect = SVC.DIALECT): string[] {
  return dialect.validate(term(spec, dialect), bound);
}

/** The term `spec` writes, as SystemVerilog source, printed from its dialect's holder. */
function text(spec: unknown, dialect: F.Dialect = SVC.DIALECT): string {
  return SVCText.ToText(held(spec, dialect));
}

/** A dialect's `Term`'s branches' schemas, by tag. */
function schemas(dialect: F.Dialect): Map<string, any> {
  return new Map(store.registered(dialect.union).branches.map((b: any) => [b.name, b.schema]));
}

const union = (dialect: F.Dialect): string[] => [...schemas(dialect).keys()]; // its Term's branches
const spelled = (spelling: string) => (i: any) => i.spelling(spelling); // an identifier, of its spelling
const name = (spelling: string) => (u: any) => u.nameexpression((n: any) => n.name((i: any) => i.identifier(spelled(spelling))));
const holds = (expression: unknown) => (u: any) => u.expressionconstraint((c: any) => c.expression(expression)); // `expression;`
const block = (...items: unknown[]) => (u: any) => u.constraintblock((b: any) => b.items(items)); // `{ items }`
const names = (set: ReadonlySet<string>): string[] => [...set].sort();
const x = name("x"), y = name("y"), a = name("a"), i = name("i"), j = name("j");

<a id="SVC-01"></a>
## SVC-01 · Its union lists Expressions' branches and its own kinds, and Assertions' a distribution; its holder is validated, linked by `HasRoot` and round-tripped

In [ ]:
// Its Term lists Expressions' branches, which that install adds, then its own kinds, then the Preprocessor's macro usage;
// Assertions' lists a distribution too, as a sequence's expression may be one (16.7), each schema one in both unions.
assert(same(SVC.TAGS, ["constraintblock", "expressionconstraint", "implicationconstraint", "conditionalconstraint",
  "foreachconstraint", "solvebeforeconstraint", "disablesoftconstraint", "uniqueconstraint", "distexpression"]));
const own = SV.TAGS.length + 2 + SVC.TAGS.length;
assert(same(union(SVC.DIALECT).slice(0, own + 1), [...SV.TAGS, "error", "missing", ...SVC.TAGS, "macrousage"]));
assert(same(union(SVA.DIALECT).slice(-2), ["distexpression", "macrousage"]) && !union(SV.DIALECT).includes("distexpression"));
assert(schemas(SVA.DIALECT).get("distexpression") === schemas(SVC.DIALECT).get("distexpression"));
assert(SV.TAGS.every((tag) => schemas(SVC.DIALECT).get(tag) === schemas(SV.DIALECT).get(tag)));
assert(same([...SVC.DIALECT.kinds().keys()], [...SVC.TAGS, "error", "missing"]) && same(SVC.HELPERS, ["DistItem"]));
const DistItem: any = store.registered("Expressions.SystemVerilog.Constraints.DistItem");
assert(!DistItem.ref && !union(SVC.DIALECT).includes("distitem")); // a helper: a value object, no kind
// Its dialect: named by its family, a branch of the framework's Term and of its kind union of dialects, a singleton.
assert(SVC.DIALECT.name() === "SystemVerilog Constraints" && SVC.DIALECT.tag === "systemverilog_constraints");
const branchOf = (union: string, tag: string) => store.registered(union).branches.find((b: any) => b.name === tag).schema;
assert(branchOf("Expressions.Framework.Term", "systemverilog_constraints") === store.registered(SVC.DIALECT.union));
assert(branchOf("Expressions.Framework.Dialect", "systemverilog_constraints") === store.registered("Expressions.SystemVerilog.Constraints.Dialect"));
assert(store.singleton("Expressions.SystemVerilog.Constraints") === store.Instances.Expressions.SystemVerilog.Constraints);
assert(same(SVC.DIALECT.reads(), FAMILY.filter((d) => d !== SVC.DIALECT)));
// Its holder: a reference object refining the framework's Expression, its root narrowed to its Term. A constraint block
// held is validated, linked by HasRoot and round-tripped through JSON and YAML.
const Holder: any = store.registered("Expressions.SystemVerilog.Constraints.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === store.registered(SVC.DIALECT.union));
const constraints = held(block(holds(SV.binary(">", x, 0n)), (u: any) => u.distexpression((d: any) => d.value(y).items([
  (e: any) => e.value(SV.literal(0n)).operator(":=").weight(SV.literal(3n))]))));
const linked = store.transaction(() => store.Schemas.OfExpression((s: any) => s.name("svc.Randomized").expression(
  (e: any) => e.insert((h: any) => h.expression(constraints)))));
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, constraints)), [true]));
assert(Validators.Validate(store)(Holder, constraints).length === 0 && SVC.DIALECT.validate(constraints, ["x", "y"]).length === 0);
assert(F.dialectOf(F.term(constraints)) === SVC.DIALECT.union);
assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, constraints)) as any).root, constraints.root));
assert(F.same((YAML.FromYAML(store)(Holder, YAML.ToYAML(store)(Holder, constraints)) as any).root, constraints.root));
const Term: any = store.registered(SVC.DIALECT.union); // its Term alone, as any union value
assert(F.same(JSON.FromJSON(store)(Term, JSON.ToJSON(store)(Term, F.term(constraints))), F.term(constraints)));
// Its root holds its Term's kinds alone: a Types term is refused by the schema, and another family's term by its
// validation, as it refuses its own to the other.
raises(Error, () => held(SVC.DIALECT.make(new F.Form("structtype", { keyword: "struct" }))), "'structtype' is not a member of a union value");
const python = F.term(held(1n, P.DIALECT));
assert(same(SVC.DIALECT.validate(python), ["a Python3 literal is not a SystemVerilog Constraints term"]));
assert(same(P.DIALECT.validate(F.term(constraints)), ["a SystemVerilog Constraints constraintblock is not a Python3 term"]));

<a id="SVC-02"></a>
## SVC-02 · A constraint block holds constraints, and an expression constraint a distribution or an expression, `soft` or not

In [ ]:
{
  // A constraint block holds its items, constraints, as its Term (mbse-programs' `Constraint | Directive | Comment`); an
  // expression constraint its expression, a distribution or an expression (`expression_or_dist`), and `soft`, a bool.
  const kinds = schemas(SVC.DIALECT);
  const Term: any = store.registered(SVC.DIALECT.union);
  assert(same([...kinds.get("constraintblock").properties.keys()], ["items"]) && kinds.get("constraintblock").properties.get("items").schema.element === Term);
  assert(same([...kinds.get("expressionconstraint").properties.keys()], ["soft", "expression"]));
  assert(kinds.get("expressionconstraint").properties.get("expression").schema === Term);
  assert(equal(F._parts(kinds.get("constraintblock")), [[], ["items"], []]) && equal(F._parts(kinds.get("expressionconstraint")), [["expression"], [], ["soft"]]));
  const soft = (u: any) => u.expressionconstraint((c: any) => c.soft(true).expression(SV.binary(">", x, 0n)));
  const cases: [unknown, string[]][] = [
    [block(), []], [block(soft, block(holds(y))), []],
    [holds((u: any) => u.distexpression((d: any) => d.value(x).items([(e: any) => e.value(SV.literal(1n))]))), []],
    [(u: any) => u.expressionconstraint((c: any) => c.soft(true)), ["an expressionconstraint needs an expression"]],
    [(u: any) => u.expressionconstraint((c: any) => c.soft("yes").expression(x)), ["an expressionconstraint's soft must be a bool, got str"]],
    [block(holds(name("z"))), ["argument 0: expression: nameexpression 'z' is not bound"]], // its items' terms, in scope
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec, ["x", "y"]), problems), problems.join("; "));
  assert(same(names(SY.free(term(block(soft, holds(y))), SVC.DIALECT)), ["x", "y"]) && F.nameOf(term(soft)) === null);
  assert(SVC.DIALECT.infer(term(soft), { x: D.Anything }) === D.Anything); // a constraint's domain is not known statically
}

<a id="SVC-03"></a>
## SVC-03 · An implication's and a conditional's condition is an expression, their bodies constraints, an alternative optional

In [ ]:
{
  // An implication's and a conditional's condition is an expression, Expressions' Term; its body, consequence and
  // alternative are constraints, its own Term; a conditional's alternative may be absent.
  const kinds = schemas(SVC.DIALECT);
  const [ETerm, Term] = [store.registered(SV.DIALECT.union), store.registered(SVC.DIALECT.union)];
  assert(same([...kinds.get("implicationconstraint").properties.keys()], ["condition", "body"]));
  assert(same([...kinds.get("conditionalconstraint").properties.keys()], ["condition", "consequence", "alternative"]));
  assert(kinds.get("implicationconstraint").properties.get("condition").schema === ETerm && kinds.get("implicationconstraint").properties.get("body").schema === Term);
  assert(same(["condition", "consequence", "alternative"].map((p) => kinds.get("conditionalconstraint").properties.get(p).schema), [ETerm, Term, Term]));
  const implies = (condition: unknown, body: unknown) => (u: any) => u.implicationconstraint((c: any) => c.condition(condition).body(body));
  const when = (condition: unknown, consequence: unknown) => (u: any) => u.conditionalconstraint((c: any) => c.condition(condition).consequence(consequence));
  const cases: [unknown, string[]][] = [
    [implies(x, holds(y)), []], [when(x, holds(y)), []],
    [(u: any) => u.conditionalconstraint((c: any) => c.condition(x).consequence(holds(y)).alternative(when(y, block()))), []],
    [(u: any) => u.implicationconstraint((c: any) => c.body(holds(y))), ["an implicationconstraint needs a condition"]],
    [(u: any) => u.implicationconstraint((c: any) => c.condition(x)), ["an implicationconstraint needs a body"]],
    [(u: any) => u.conditionalconstraint((c: any) => c.condition(x)), ["a conditionalconstraint needs a consequence"]],
    [implies(name("z"), holds(y)), ["condition: nameexpression 'z' is not bound"]],
    [when(x, holds(name("z"))), ["consequence: expression: nameexpression 'z' is not bound"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec, ["x", "y"]), problems), problems.join("; "));
  // An expression subterm is of Expressions' terms alone: a constraint there is refused by the schema.
  raises(Error, () => held(SVC.DIALECT.make(new F.Form("implicationconstraint", {}, [block(), holds(y)]))),
    "'constraintblock' is not a member of a union value");
}

<a id="SVC-04"></a>
## SVC-04 · A `foreach` refines `Let`: its array its value, its body its body, binding its loop variables within its body alone

In [ ]:
{
  // A loop binds its variables within its body (18.5.8.1): it refines the framework's Let (Q8, "(a)"), its array Let's value,
  // an Expressions term, its body Let's body, its own Term; Let's name stays unset, its kind's scopes saying what it binds.
  // Its variables are each an identifier, or an empty argument for a dimension it skips (mbse-programs'
  // `Identifier | EmptyArgument`), its union LoopVariable.
  const Foreach = schemas(SVC.DIALECT).get("foreachconstraint");
  assert(Foreach.refines === store.registered("Expressions.Framework.Let") && F.isBinder(Foreach));
  assert(same([...Foreach.properties.keys()], ["name", "value", "body", "variables"])); // Let's, then its own
  assert(equal(F._parts(Foreach), [["value", "body"], [], ["name", "variables"]]));
  assert(Foreach.properties.get("value").schema === store.registered(SV.DIALECT.union));
  assert(Foreach.properties.get("body").schema === store.registered(SVC.DIALECT.union));
  const LoopVariable: any = store.registered("Expressions.SystemVerilog.Constraints.LoopVariable");
  assert(Foreach.properties.get("variables").schema.element === LoopVariable);
  assert(same(LoopVariable.branches.map((b: any) => b.name), ["identifier", "emptyargument"]));
  assert(LoopVariable.branches[0].schema === store.registered("Expressions.SystemVerilog.Expressions.Identifier"));
  assert(SVC.DIALECT.kinds().get("foreachconstraint")!.role === F.BINDING);
  const loop = (array: unknown, variables: unknown[], body: unknown) => (u: any) => u.foreachconstraint((f: any) => f.value(array).variables(variables).body(body));
  const at = (value: unknown, index: unknown) => (u: any) => u.indexexpression((e: any) => e.value(value).index(index));
  const skipped = (v: any) => v.emptyargument((e: any) => e);
  const each = term(loop(a, [(v: any) => v.identifier(spelled("i")), skipped, (v: any) => v.identifier(spelled("j"))],
    holds(SV.binary("<", at(a, i), j))));
  assert(same(names(SY.free(each, SVC.DIALECT)), ["a"]) && SVC.DIALECT.validate(each, ["a"]).length === 0 && F.nameOf(each) === null);
  assert(same(names(SVC.loopVariables(F.kindOf(each)[1])), ["i", "j"]));
  assert(equal(SVC.DIALECT.kinds().get("foreachconstraint")!.scopes!(F.kindOf(each)[1]).map(names), [[], ["i", "j"]]));
  assert(SVC.DIALECT.infer(each, { a: D.Anything }) === D.Anything);
  const cases: [unknown, string[]][] = [
    [loop(at(a, i), [(v: any) => v.identifier(spelled("i"))], holds(i)), ["value: index: nameexpression 'i' is not bound"]], // not in its array
    [loop(a, [(v: any) => v.identifier(spelled("1i"))], block()), ["'1i' is not an identifier"]],
    [loop(a, [(v: any) => v.identifier((s: any) => s)], holds(i)), ["body: expression: nameexpression 'i' is not bound"]], // no spelling: binds none
    [loop(a, [skipped], holds(i)), ["body: expression: nameexpression 'i' is not bound"]],
    [(u: any) => u.foreachconstraint((f: any) => f.body(block())), ["a foreachconstraint needs a value"]],
    [(u: any) => u.foreachconstraint((f: any) => f.value(a)), ["a foreachconstraint needs a body"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec, ["a"]), problems), problems.join("; "));
}

<a id="SVC-05"></a>
## SVC-05 · `solve ... before` orders two lists of variables, `disable soft` names its target, and `unique` its set of expressions and ranges

In [ ]:
// `disable soft` names its target, an expression; `unique` its set, expressions and ranges (IEEE 1800-2012, 18.5.5).
const span = (u: any) => u.valuerange((r: any) => r.left(SV.literal(1n)).right(SV.literal(3n))); // [1:3]
{
  // `solve ... before` holds two lists of expressions, the variables solved first and those after (18.5.10), two
  // containers of terms, each read apart; `disable soft` names its target, an expression; `unique` its set,
  // expressions and ranges (IEEE 1800-2012, 18.5.5).
  const kinds = schemas(SVC.DIALECT);
  const ETerm: any = store.registered(SV.DIALECT.union);
  assert(same([...kinds.get("solvebeforeconstraint").properties.keys()], ["solve", "before"]));
  assert(["solve", "before"].every((p) => kinds.get("solvebeforeconstraint").properties.get(p).schema.element === ETerm));
  assert(equal(F._parts(kinds.get("solvebeforeconstraint")), [[], ["solve", "before"], []]));
  const order = (solve: unknown[], before: unknown[]) => (u: any) => u.solvebeforeconstraint((s: any) => s.solve(solve).before(before));
  const node = F.node(...F.kindOf(term(order([x, y], [a]))));
  assert(same(node.lists.get("solve")!.map((t) => (F.form(t).attributes.get("name") as F.Part).attributes.get("spelling")), ["x", "y"])
    && node.lists.get("before")!.length === 1);
  const sizes = F.form(term(order([x, y], [a]))).attributes;
  assert(same([...sizes].map(([k, p]) => `${k} ${(p as F.Part).kind} ${(p as F.Part).attributes.size} ${(p as F.Part).size}`),
    ["solve solve 0 2", "before before 0 1"]));
  assert(F.same(term(SVC.DIALECT.make(F.form(term(order([x, y], [a]))))), term(order([x, y], [a]))));
  assert(same([...kinds.get("disablesoftconstraint").properties.keys()], ["target"]) && kinds.get("disablesoftconstraint").properties.get("target").schema === ETerm);
  assert(same([...kinds.get("uniqueconstraint").properties.keys()], ["set"]) && kinds.get("uniqueconstraint").properties.get("set").schema.element === ETerm);
  assert(equal(F._parts(kinds.get("uniqueconstraint")), [[], ["set"], []]));
  const cases: [unknown, string[]][] = [
    [order([x, y], [a]), []], [order([], []), []],
    [order([x], [name("z")]), ["before 0: nameexpression 'z' is not bound"]],
    [(u: any) => u.disablesoftconstraint((d: any) => d.target(x)), []],
    [(u: any) => u.uniqueconstraint((q: any) => q.set([x, a, span])), []], [(u: any) => u.uniqueconstraint((q: any) => q.set([])), []],
    [(u: any) => u.disablesoftconstraint((d: any) => d), ["a disablesoftconstraint needs a target"]],
    [(u: any) => u.uniqueconstraint((q: any) => q.set([x, name("z")])), ["argument 1: nameexpression 'z' is not bound"]],
    [(u: any) => u.uniqueconstraint((q: any) => q.set([(v: any) => v.valuerange((r: any) => r.left(x).operator("+-").right(y))])),
      ["argument 0: ValueRange.operator cannot be '+-'"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec, ["x", "y", "a"]), problems), problems.join("; "));
}

<a id="SVC-06"></a>
## SVC-06 · A distribution's items, each a value or `default` with an operator and a weight, checked as mbse-programs checks them; Assertions' subterms may be one

In [ ]:
// A distribution (18.5.4): its value an expression, its items helpers, each its value (an expression or a range), or none
// for `default`, and its operator and weight, both or neither, as mbse-programs checks them.
const dist = (...items: unknown[]) => (u: any) => u.distexpression((d: any) => d.value(x).items(items));
const item = (value: unknown = null, operator: string | null = null, weight: unknown = null) => (e: any) => // an item, of what it has
  ([["value", value], ["operator", operator], ["weight", weight]] as [string, unknown][]).filter(([, part]) => part !== null)
    .reduce((b: any, [key, part]) => b[key](part), e);
{
  const Dist = schemas(SVC.DIALECT).get("distexpression");
  const ETerm: any = store.registered(SV.DIALECT.union);
  assert(same([...Dist.properties.keys()], ["value", "items"]) && Dist.properties.get("value").schema === ETerm);
  assert(Dist.properties.get("items").schema.element === DistItem && equal(F._parts(Dist), [["value"], [], ["items"]]));
  assert(same([...DistItem.properties.keys()], ["value", "operator", "weight"]));
  assert(DistItem.properties.get("value").schema === ETerm && DistItem.properties.get("weight").schema === ETerm && same(SVC.OPERATORS, [":=", ":/"]));
  const cases: [unknown, string[]][] = [
    [dist(item(SV.literal(0n), ":=", SV.literal(40n)), item(span, ":/", SV.literal(60n)), item(y), item(null, ":/", SV.literal(1n))), []],
    [dist(item(SV.literal(0n), "=", SV.literal(1n))), ["DistItem.operator cannot be '='"]],
    [dist(item(SV.literal(0n), ":=")), ["a DistItem has both an operator and a weight, or neither"]],
    [dist(item(SV.literal(0n), null, SV.literal(2n))), ["a DistItem has both an operator and a weight, or neither"]],
    [dist(item()), ["a default DistItem weighs its values with :/"]],
    [dist(item(null, ":=", SV.literal(1n))), ["a default DistItem weighs its values with :/"]],
    [dist(item(SV.literal(0n), ":=", name("w"))), ["items 0 weight: nameexpression 'w' is not bound"]],
    [(u: any) => u.distexpression((d: any) => d.items([item(y)])), ["a distexpression needs a value"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec, ["x", "y"]), problems), problems.join("; "));
  // Assertions' Term lists it: a sequence's expression, a call's argument (`Property | Expression`), may be a distribution.
  const weighted = dist(item(SV.literal(1n), ":=", SV.literal(2n)));
  assert(validate(weighted, ["x"], SVA.DIALECT).length === 0 && F.dialectOf(term(weighted, SVA.DIALECT)) === SVA.DIALECT.union);
  const called = term(SV.call("f", weighted), SV.DIALECT);
  assert(F.dialectOf(F.form(called).arguments[1]) === SVA.DIALECT.union && SV.DIALECT.validate(called, ["f", "x"]).length === 0);
  assert(SVA.DIALECT.kind(F.form(called).arguments[1])[2] === SVC.DIALECT.kinds().get("distexpression")); // read as its dialect reads it
  raises(Error, () => held(SV.DIALECT.make(new F.Form("distexpression", {}, [x])), SV.DIALECT), // Expressions' Term lists it not
    "'distexpression' is not a member of a union value");
}

<a id="SVC-07"></a>
## SVC-07 · A tree mixing constraints and expressions, held by `randomize() with` or by its own holder, validated and printed from either, each dialect of the family reading it

In [ ]:
{
  // A tree mixing its kinds with expressions: `randomize() with` holds constraints, its own Term, which hold expressions,
  // Expressions' Term, and a distribution; validated, its free names found and printed from Expressions' holder, through
  // the property that holds its Term as a subterm, and from its own holder, each dialect reading the others' terms.
  const randomize = (...items: unknown[]) => (u: any) => u.randomizewithexpression((r: any) => r.call(SV.method(x, "randomize")).items(items));
  const loop = (u: any) => u.foreachconstraint((f: any) => f.value(a).variables([(v: any) => v.identifier(spelled("i"))]).body(
    holds((d: any) => d.distexpression((e: any) => e.value((q: any) => q.indexexpression((s: any) => s.value(a).index(i))).items([
      (t: any) => t.value(SV.literal(0n)).operator(":=").weight(name("w"))])))));
  const mixed = term(randomize(loop, (u: any) => u.implicationconstraint((c: any) => c.condition(SV.inside(y, SV.span(1n, 3n))).body(
    holds(SV.binary("==", name("z"), 0n))))), SV.DIALECT);
  assert(SV.DIALECT.validate(mixed, ["x", "y", "a", "w", "z"]).length === 0 && same(names(SY.free(mixed, SV.DIALECT)), ["a", "w", "x", "y", "z"]));
  const items = F.form(mixed).arguments.slice(1);
  assert(same(items.map((t: any) => F.dialectOf(t)), [SVC.DIALECT.union, SVC.DIALECT.union]) && SV.DIALECT.kind(items[0]!)[2] === SVC.DIALECT.kinds().get("foreachconstraint"));
  assert(same(SV.DIALECT.validate(mixed, ["x", "y", "a", "z"]), [
    "argument 0: body: expression: items 0 weight: nameexpression 'w' is not bound"]));
  assert(SVText.ToText(mixed) === "x.randomize() with { foreach (a[i]) a[i] dist {0 := w}; y inside {[1:3]} -> z == 0; }");
  assert(SVC.DIALECT.infer(items[0]!, { a: D.Anything, w: D.Anything }) === D.Anything);
  // From its own holder: a block whose items hold expressions, and an expression whose argument is its own term again
  const nested = held(block(holds((u: any) => u.callexpression((c: any) => c.callee(name("f")).arguments([(d: any) => d.distexpression(
    (e: any) => e.value(x).items([(t: any) => t.value(SV.literal(1n))]))]))), randomize(holds(y))));
  assert(SVC.DIALECT.validate(nested, ["f", "x", "y"]).length === 0 && same(names(SY.free(F.term(nested), SVC.DIALECT)), ["f", "x", "y"]));
  assert(SVCText.ToText(nested) === "{ f(x dist {1}); x.randomize() with { y; }; }");
  assert(FAMILY.every((d) => d.validate(F.term(nested), ["f", "x", "y"]).length === 0)); // each dialect of the family reads it alike
  assert(same(SVC.DIALECT.validate(F.term(nested), ["x", "y"]), ["argument 0: expression: callee: nameexpression 'f' is not bound"]));
}

<a id="SVC-08"></a>
## SVC-08 · Invalid syntax stands wherever a constraint or an expression does: reported, printed as written, left in the residual by simplification, named

In [ ]:
{
  // Invalid syntax, `error` and `missing`, stands wherever its Term or Expressions' does: reported, printed as written,
  // refused by evaluation by name.
  const error = (written: string) => (u: any) => u.error((e: any) => e.text(written));
  const missing = (u: any) => u.missing((m: any) => m.kind("Constraint"));
  const cases: [unknown, string[], string][] = [
    [error("x >"), ["invalid syntax 'x >'"], "x >"],
    [block(holds(x), missing), ["argument 1: missing Constraint"], "{ x;  }"],
    [(u: any) => u.implicationconstraint((c: any) => c.condition(error("a &")).body(holds(x))), ["condition: invalid syntax 'a &'"], "a & -> x;"],
    [(u: any) => u.foreachconstraint((f: any) => f.value(a).body(error("??"))), ["body: invalid syntax '??'"], "foreach (a[]) ??"],
    [(u: any) => u.distexpression((d: any) => d.value(x).items([(t: any) => t.value(error("1:")).operator(":=").weight(SV.literal(1n))])),
      ["items 0 value: invalid syntax '1:'"], "x dist {1: := 1}"],
  ];
  for (const [spec, problems, source] of cases) {
    assert(same(validate(spec, ["x", "a"]), problems), problems.join("; "));
    assert(text(spec) === source, source);
  }
  raises(NotEvaluated, () => SVV.OfAny(term(error("x >")), {}).evaluate(), "SystemVerilog Constraints error is not simplified"); // invalid syntax stays, named
  raises(ValueError, () => text((u: any) => u.error((e: any) => e)), "SystemVerilog Constraints error has no text to print");
}

<a id="SVC-09"></a>
## SVC-09 · The editions it covers are data: IEEE Std 1800-2005 to 1800-2023, each with its own kinds from the edition it arrived in

In [ ]:
{
  // Its editions are data (CAP-268): IEEE Std 1800-2005 to 1800-2023, owned by designation, each with its own kinds it has,
  // from mbse-programs' SINCE: `disable soft` and `unique` from IEEE 1800-2012, the others from 1800-2005; its define runs
  // again writing nothing, and binds each edition by its common name spelled as an identifier.
  const singleton = store.Instances.Expressions.SystemVerilog.Constraints;
  const LABELS = ["SystemVerilog-2005", "SystemVerilog-2009", "SystemVerilog-2012", "SystemVerilog-2017", "SystemVerilog-2023"];
  assert(same([...FS.standards(singleton).keys()], [2005, 2009, 2012, 2017, 2023].map((year) => `IEEE Std 1800-${year}`)));
  assert(equal(SVC.SINCE, map({ disablesoftconstraint: 5, uniqueconstraint: 5 })) && SV.EDITIONS[5]![0] === "IEEE Std 1800-2012");
  assert(same(FS.tags(FS.find(singleton, "SystemVerilog-2012")!), ["IEEE Std 1800-2012", "SystemVerilog-2012"]));
  assert(equal(FS.properties(FS.find(singleton, "SystemVerilog-2005")!), map({ bodies: ["org.ieee"], number: "1800",
    revision: ["year", 2005n], names: ["SystemVerilog-2005"] })));
  assert(FS.find(singleton, "Verilog-2005") === null); // IEEE 1364's have no constraints
  const editions = new Map([...schemas(SVC.DIALECT)].map(([tag, schema]) => [tag, [...FS.of(singleton, schema).values()].map((s) => FS.tags(s).at(-1))]));
  assert(union(SVC.DIALECT).every((tag) => same(editions.get(tag)!, SVC.TAGS.includes(tag) ? LABELS.slice((SVC.SINCE.get(tag) ?? 3) - 3) : [])));
  assert(same(FS.kinds(FS.find(singleton, "SystemVerilog-2009")!), SVC.TAGS.filter((tag) => !SVC.SINCE.has(tag)).map((tag) => schemas(SVC.DIALECT).get(tag))));
  const frame = store.transaction(SVConstraints.define); // again: nothing new
  assert(frame[0].get("SystemVerilog_2023") === FS.find(singleton, "SystemVerilog-2023") && FS.standards(singleton).size === 5);
}

<a id="SVC-10"></a>
## SVC-10 · Printing every kind as `randomize() with` writes it, on one line, with IEEE 1800's precedence

In [ ]:
{
  // Printing ports mbse-programs' printer (`constraint_text`, `DistExpression`): a constraint on one line, as
  // `randomize() with` writes it; an implication's condition parenthesized below `->`'s precedence, a distribution's value
  // below a relational operator's; a distribution binds as `inside` does, and held as a constraint it is ended by `;`.
  const xGt = SV.binary(">", x, 0n);
  const cases: [unknown, string][] = [
    [block(), "{}"], [block(holds(xGt), holds(y)), "{ x > 0; y; }"], [block(block(holds(x))), "{ { x; } }"],
    [(u: any) => u.expressionconstraint((c: any) => c.soft(true).expression(xGt)), "soft x > 0;"],
    [(u: any) => u.expressionconstraint((c: any) => c.soft(false).expression(xGt)), "x > 0;"],
    [(u: any) => u.implicationconstraint((c: any) => c.condition(SV.binary("||", x, y)).body(holds(y))), "x || y -> y;"],
    [(u: any) => u.implicationconstraint((c: any) => c.condition(SV.binary("->", x, y)).body(block(holds(y)))), "(x -> y) -> { y; }"],
    [(u: any) => u.implicationconstraint((c: any) => c.condition(SV.conditional(x, y, 0n)).body(holds(y))), "x ? y : 0 -> y;"],
    [(u: any) => u.conditionalconstraint((c: any) => c.condition(x).consequence(holds(y))), "if (x) y;"],
    [(u: any) => u.conditionalconstraint((c: any) => c.condition(x).consequence(block(holds(y))).alternative(
      (v: any) => v.conditionalconstraint((d: any) => d.condition(y).consequence(holds(x)).alternative(holds(a))))),
    "if (x) { y; } else if (y) x; else a;"],
    [(u: any) => u.foreachconstraint((f: any) => f.value(SV.member(a, "q")).variables([(v: any) => v.emptyargument((e: any) => e),
      (v: any) => v.identifier(spelled("j"))]).body(holds(j))), "foreach (a.q[, j]) j;"],
    [(u: any) => u.foreachconstraint((f: any) => f.value(a).body(block())), "foreach (a[]) {}"],
    [(u: any) => u.solvebeforeconstraint((s: any) => s.solve([x, y]).before([SV.member(a, "q")])), "solve x, y before a.q;"],
    [(u: any) => u.disablesoftconstraint((d: any) => d.target(SV.member(a, "q"))), "disable soft a.q;"],
    [(u: any) => u.uniqueconstraint((q: any) => q.set([x, a, span])), "unique {x, a, [1:3]};"],
    [(u: any) => u.uniqueconstraint((q: any) => q.set([])), "unique {};"],
    [dist(item(SV.literal(0n), ":=", SV.literal(40n)), item(span, ":/", SV.binary("+", x, 1n)), item(y), item(null, ":/", SV.literal(1n))),
      "x dist {0 := 40, [1:3] :/ x + 1, y, default :/ 1}"],
    [(u: any) => u.distexpression((d: any) => d.value(SV.binary("+", x, y)).items([item(y)])), "x + y dist {y}"],
    [(u: any) => u.distexpression((d: any) => d.value(SV.binary("<", x, y)).items([item(y)])), "x < y dist {y}"],
    [(u: any) => u.distexpression((d: any) => d.value(SV.binary("==", x, y)).items([item(y)])), "(x == y) dist {y}"],
    [block(dist(item(y)), holds(dist(item(y)))), "{ x dist {y}; x dist {y}; }"], // held as a constraint, or by one
  ];
  for (const [spec, source] of cases) assert(text(spec) === source, `${source} != ${text(spec)}`);
  // A distribution's precedence is a relational operator's, where an Assertions' operand holds it (an expression's
  // operands, Expressions' terms, hold none).
  const weighted = SVText.printer().node(term(dist(item(y))));
  assert(SVText.printer().precedence(weighted, "x dist {y}") === SVText.RELATIONAL);
  assert(SVText.printer().operand(term(dist(item(y))), SVText.RELATIONAL + 1) === "(x dist {y})");
  assert([SVConstraints, SystemVerilog].every((pkg) => pkg.Text.ToText === SVText.ToText)); // one printer for the family
}

<a id="SVC-11"></a>
## SVC-11 · Simplification leaves each of its kinds in the residual, named

In [ ]:
{
  // Its kinds are not evaluated (Q11): SystemVerilog's evaluator refuses each by name, from any holder, valid as it is.
  const samples = new Map<string, unknown>([
    ["constraintblock", block(holds(SV.literal(1n)))],
    ["expressionconstraint", holds(SV.literal(1n))],
    ["implicationconstraint", (u: any) => u.implicationconstraint((c: any) => c.condition(SV.literal(1n)).body(block()))],
    ["conditionalconstraint", (u: any) => u.conditionalconstraint((c: any) => c.condition(SV.literal(1n)).consequence(block()))],
    ["foreachconstraint", (u: any) => u.foreachconstraint((f: any) => f.value(SV.literal(1n)).body(block()))],
    ["solvebeforeconstraint", (u: any) => u.solvebeforeconstraint((s: any) => s.solve([SV.literal(1n)]).before([SV.literal(2n)]))],
    ["disablesoftconstraint", (u: any) => u.disablesoftconstraint((d: any) => d.target(SV.literal(1n)))],
    ["uniqueconstraint", (u: any) => u.uniqueconstraint((q: any) => q.set([SV.literal(1n)]))],
    ["distexpression", (u: any) => u.distexpression((d: any) => d.value(SV.literal(1n)).items([(t: any) => t.value(SV.literal(1n))]))],
  ]);
  assert(same([...samples.keys()], SVC.TAGS));
  for (const [tag, spec] of samples) {
    assert(validate(spec).length === 0, tag);
    raises(NotEvaluated, () => SVV.OfAny(term(spec), {}).evaluate(), `SystemVerilog Constraints ${tag} is not simplified`);
  }
  raises(NotEvaluated, () => SVV.OfAny(term(SV.call("f", samples.get("distexpression")), SV.DIALECT), // within a call
    new SVV.Scope({}, { functions: { f: (value: unknown) => value } })).evaluate(), "SystemVerilog Constraints distexpression is not simplified");
}